# Generate LLM class descriptions (frozen artifacts)

Writes `config/descriptions/{dataset}_{style}.json` for every
`(DATASETS x STYLES)` combination in one run, then zips the whole
`config/descriptions/` directory so it can be downloaded once and unzipped
straight into the repo working copy — these are small text files meant to
be **committed to git**, not published as a Kaggle Dataset like every other
notebook here.

**Deliberately NOT `§2.0`'s "one run = one config = one zip".** Every other
notebook in this project produces a GB-scale GPU cache, where a second axis
means a second multi-hour session; this one is a CPU-only loop over a
handful of tiny text API calls, so looping over every `(dataset, style)`
pair inside one run is strictly more convenient and costs nothing extra —
API rate limits, not GPU-hours, are the constraint here. Each (dataset,
style) pair still gets its own file (`{dataset}_{style}.json`), so nothing
about downstream readers changes: `extract_vlm_features.ipynb` and
`run_al_main.ipynb` still read exactly one file per (dataset, style), the
same as before this notebook swept.

**This is a frozen artifact, not a reproducible computation.** A hosted
model call is not bit-for-bit reproducible even at `temperature=0.0` --
Google's own docs do not guarantee determinism across requests, let alone
across months as the served weights behind a fixed model name change, and a
pinned model name can itself be retired without notice (`gemini-2.5-flash`,
an earlier default here, returned `404 NOT_FOUND` for new callers as of
2026-08). What is reproducible is the **written JSON file** committed to
git, not "re-run this notebook and expect the same text" -- each file
records `model`, `temperature`, `seed`, `generated_at` and a `sha256` of the
text, which is proof of what the model returned once, not a promise it will
again. If a description needs to change, regenerate deliberately
(`OVERWRITE=True`) and commit the new file.

Runs on CPU -- no GPU needed, no dataset image attached. Kept as a notebook
(rather than a local script) only for consistency with every other pipeline
stage in this project.

**Two styles**: `llm_short` (one dense sentence) and `llm_morphology` (2-4
sentences on cell/nuclear shape, arrangement, texture, staining). A third
style, `llm_multi` (N differently-phrased variants per class), was removed
from the whole project -- recoverable from git history if a multi-variant
ensemble is needed again.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"])

if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
# Every (dataset, style) combination below runs in this ONE session --
# 3 datasets x 2 styles = 6 API sweeps, 6 files, one at the end (see the
# intro cell for why this notebook does not follow the usual one-run-one-
# config rule every GPU notebook in this project follows).
DATASETS = ["pathmnist", "histoset", "skintissue"]
STYLES = ["llm_short", "llm_morphology"]

# Pick whatever model is CURRENTLY AVAILABLE to your API key -- gemini-2.5-flash
# (an earlier default here) was retired for new callers as of 2026-08 (404
# NOT_FOUND, "no longer available to new users"). No model family is
# blocked: each written file records `model`, `temperature`, `seed`,
# `generated_at` and a `sha256` of the actual text returned, which is proof
# the model produced exactly this text once -- not a promise that
# TEMPERATURE reproduces it on a later call, or that this exact model name
# will still resolve when someone re-runs this later. gemini-3.7-flash is
# the strongest model actually reachable on the free tier as of 2026-08:
# every Pro-tier model (2.5 Pro, 3.1 Pro) is paid-only on the API, even
# though Pro is free to try interactively in AI Studio.
MODEL = "gemini-3.7-flash"
TEMPERATURE = 0.0
SEED = 42
OVERWRITE = False              # refuse to clobber an existing frozen file unless True
API_KEY = ""                   # Kaggle Secret (GEMINI_API_KEY) or pasted directly

In [ ]:
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
        print("[auth] Loaded API_KEY from Kaggle Secret 'GEMINI_API_KEY'.")
    except Exception as exc:
        print(f"[auth] No Kaggle Secret 'GEMINI_API_KEY' found ({exc}).")
        print("       Paste a key into API_KEY above, or add a Kaggle Secret named")
        print("       GEMINI_API_KEY (Add-ons -> Secrets) before running this cell.")
else:
    print("[auth] Using API_KEY from the EDIT cell.")

assert API_KEY, "No API key available -- set API_KEY or a Kaggle Secret GEMINI_API_KEY"

In [ ]:
import json

import yaml

from features.descriptions import description_path, generate_descriptions

In [ ]:
assert isinstance(DATASETS, list) and DATASETS, "DATASETS must be a non-empty list"
assert isinstance(STYLES, list) and STYLES, "STYLES must be a non-empty list"
for ds in DATASETS:
    assert ds in ("pathmnist", "histoset", "skintissue"), f"unknown dataset {ds!r}"
for st in STYLES:
    assert st in ("llm_short", "llm_morphology"), f"unknown style {st!r}"

with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

print(f"model: {MODEL} | temperature: {TEMPERATURE} | seed: {SEED}")
print(f"sweep: {len(DATASETS)} datasets x {len(STYLES)} styles = "
      f"{len(DATASETS) * len(STYLES)} files")
print()
print("NOTE: a hosted model call is not bit-for-bit reproducible even at temperature=0.0.")
print("      Each written JSON file is the reproducible artifact -- not the act of")
print("      calling the API. Re-running this notebook later may produce different")
print("      text even with identical settings; that is why OVERWRITE defaults to False.")

results = []  # (dataset, style, out_path, status) for the summary table at the end

In [ ]:
for dataset in DATASETS:
    dataset_info = config["datasets"][dataset]
    class_names = list(dataset_info["descriptions"])

    for style in STYLES:
        out_path = Path(description_path(dataset, style))
        print("=" * 70)
        print(f"{dataset} / {style} -> {out_path}")

        if out_path.is_file() and not OVERWRITE:
            print(f"  SKIPPED: {out_path} already exists (this is a frozen artifact, "
                  "committed to the repo). Set OVERWRITE=True to regenerate deliberately "
                  "-- this changes the file's sha256 and invalidates every cached text "
                  "prototype built from the old text.")
            results.append((dataset, style, str(out_path), "skipped (exists)"))
            continue

        payload = generate_descriptions(
            dataset=dataset,
            style=style,
            class_names=class_names,
            model=MODEL,
            temperature=TEMPERATURE,
            seed=SEED,
            api_key=API_KEY,
        )

        # Verify before writing: every class has a non-empty description, and
        # the class order matches config.yaml exactly -- the same order
        # extract_vlm_features.ipynb and every downstream reader assumes.
        assert list(payload["descriptions"]) == class_names, (
            f"{dataset}/{style}: class order drifted from config.yaml"
        )
        for name, text in payload["descriptions"].items():
            assert isinstance(text, str) and text.strip(), f"{dataset}/{style}/{name}: empty description"

        for name, text in payload["descriptions"].items():
            print(f"  [{name}] {text}")
        print(f"  sha256: {payload['sha256']}")

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, sort_keys=True, ensure_ascii=False)
        print(f"  wrote {out_path} ({out_path.stat().st_size} bytes)")
        results.append((dataset, style, str(out_path), "written"))

print("=" * 70)
print("\nSummary:")
for dataset, style, path, status in results:
    print(f"  {dataset:12} {style:16} {status:18} {path}")

In [ ]:
# Zip the WHOLE config/descriptions/ directory -- not just the files this
# run wrote, so a re-run that only regenerated some (dataset, style) pairs
# still ships every frozen description alongside them, including this
# notebook's own README.md documenting the directory. Unlike every other
# notebook here, this is not a Kaggle-Dataset-bound cache: these are small
# text files meant to be committed to git, so the zip is just a convenient
# single download to unzip straight into the repo working copy.
import shutil

DESCRIPTIONS_DIR = Path("config/descriptions")
assert DESCRIPTIONS_DIR.is_dir(), f"nothing to archive at {DESCRIPTIONS_DIR}"

WORKING = Path("/kaggle/working")
ARCHIVE = WORKING / "class_descriptions"
shutil.make_archive(str(ARCHIVE), "zip", root_dir=DESCRIPTIONS_DIR)
size_kb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e3

print(f"{ARCHIVE.name}.zip  ({size_kb:.1f} KB) contains:")
for path in sorted(DESCRIPTIONS_DIR.iterdir()):
    if path.is_file():
        print(f"    {path.name}  ({path.stat().st_size} bytes)")

print(f"""
NEXT STEPS
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. Unzip it directly into this repo's config/descriptions/ directory
     (each file inside is already named {{dataset}}_{{style}}.json, matching
     description_path() -- no renaming needed).
  3. Commit the new/changed files -- these are small text files, meant to be
     committed to git like config/prompts/, unlike every GPU-cache notebook's
     Kaggle-Dataset zip.
  4. extract_vlm_features.ipynb / run_al_main.ipynb with a matching
     DESCRIPTION_STYLE will then find them.""")